<img src="https://raw.githubusercontent.com/tancao1128/ams261-labs/main/assets/lab13-banner.png" alt="AMS 261 Lab 13 · Hill Survey: the contour map of the hill z = sqrt(16 − 4x² − y²) at heights 0, 2 and 4, and its profiles through the summit" width="100%">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tancao1128/ams261-labs/blob/main/labs/AMS261-Lecture13.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/tancao1128/ams261-labs/main?labpath=labs%2FAMS261-Lecture13.ipynb)
· [Download this notebook (.ipynb)](https://raw.githubusercontent.com/tancao1128/ams261-labs/main/labs/AMS261-Lecture13.ipynb)

## 0 · Mission briefing

A survey drone maps one hill. The height of the ground at the map point $(x,y)$ is

$$
z=f(x,y)=\sqrt{16-4x^2-y^2}.
$$

The numbers $x$ and $y$ are in kilometers. The height $z$ is in hundreds of meters.
The drone must answer three questions.

1. At which map points can it read a height? (the **domain**)
2. Which heights can it read? (the **range**)
3. What do the contour lines at heights $0$, $2$ and $4$ look like? (the **level curves**)

In this lab we answer all three with sympy and with pictures.

- **Time:** about 30 minutes.
- **When:** after class, before Lecture 14.
- **Before you start:** read §4, §7 and §8 of the Lecture 13 page, up to Example 10.
- **Section numbers.** Sections are numbered 0–13. "(lecture §n)" points to the matching section of the Lecture 13 page.

**Lab objectives.** By the end of this lab you can

1. find the domain of $f$ by listing its restriction, and draw it as a region of the plane;
2. find the range of $f$ by asking which heights occur;
3. find the level curves $f(x,y)=k$, check $k$ against the range, and read steepness from their spacing.

**Directions on the map.** East is the positive $x$-direction. North is the positive $y$-direction.
The summit is the map point $(0,0)$.

**How to work.** Run the cells from top to bottom with Shift+Enter. A cell marked ✏️ asks you to type something first.

> **Not graded. Practice only.**

In [ ]:
# SETUP: run this cell first. It needs no local files.
import base64, json, math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.collections import LineCollection
from IPython.display import HTML, Markdown, display

try:
    from ipywidgets import (SelectionSlider, VBox, Layout,
                            HTML as WidgetHTML, interactive_output)
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

x, y, z = sp.symbols('x y z', real=True)

# Colour tokens. Each colour has one meaning in every plot of this lab.
INK = '#1E293B'        # text, the summit, the contour at height 4
GRAD = '#2563EB'       # primary: the contour at height 2, and the domain
DESCENT = '#DC2626'    # negative: points with no reading (outside the domain)
ZERO = '#059669'       # zero: the contour at height 0 (the edge of the domain)
DIRECTION = '#D97706'  # highlight: the height k you choose
CONTOUR = '#94A3B8'    # other level curves
SLICE = '#7C3AED'      # slices and profiles of the surface

# Console tokens. They are used only for the frame around each plot.
HUD_NAVY = '#0A0E14'   # top and bottom bars
HUD_PANEL = '#0F1A2B'  # the outer frame
CYAN = '#22D3EE'       # frame text and corner brackets
SCREEN = '#F8FAFC'     # the light screen behind every panel

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})


def show_math(latex):
    # Display one formula.
    display(Markdown(f"$${latex}$$"))


def show_text(text):
    # Display formatted text.
    display(Markdown(text))


def fmt(v, digits=2):
    # Format a number; never print -0.00.
    v = float(v)
    if abs(v) < 0.5 * 10 ** (-digits):
        v = 0.0
    return f"{v:.{digits}f}"


def legend_line(color, label, ls='-', marker=None):
    # A legend entry for an arrow or a marker.
    return Line2D([0], [0], color=color, ls=ls, lw=2.2 if marker is None else 0,
                  marker=marker, markersize=8, label=label)


def as_number(v):
    # Accept a number or a string such as "2*sqrt(2)".
    return float(sp.sympify(v))


def check_answer(given, target, tol=5e-3):
    # Compare an answer with a target. Returns True or False; never raises.
    try:
        if given is None:
            return False
        if isinstance(target, bool):
            return given is target
        if isinstance(target, (list, tuple)):
            return (len(given) == len(target)
                    and all(check_answer(a, b, tol) for a, b in zip(given, target)))
        return math.isclose(as_number(given), float(target), rel_tol=1e-3, abs_tol=tol)
    except Exception:
        return False


def hud_frame(fig, label, footer='HILL SURVEY · z = √(16 − 4x² − y²) · x, y in km · z in 100 m'):
    # Put a finished figure inside a navy console frame.
    # All panels, labels and legends sit on one light screen, so axis text stays dark on light.
    side, top, foot, gap, pad = 0.30, 0.40, 0.30, 0.10, 0.12   # sizes in inches
    w, h = fig.get_size_inches()
    W, H = w + 2 * (side + pad), h + top + foot + 2 * (gap + pad)
    fig.set_size_inches(W, H)
    fig.patch.set_facecolor(HUD_PANEL)

    # Dark ink on the light screen.
    for ax in fig.axes:
        ax.set_facecolor(SCREEN)
        ax.tick_params(colors=INK, labelcolor=INK)
        ax.title.set_color(INK)
        ax.xaxis.label.set_color(INK)
        ax.yaxis.label.set_color(INK)
        for spine in ax.spines.values():
            spine.set_color('#64748B')
        if ax.get_legend() is not None:
            for t in ax.get_legend().get_texts():
                t.set_color(INK)

    # The screen, and the area inside it where the panels go.
    sx0, sy0, sx1, sy1 = side, foot + gap, W - side, H - top - gap   # in inches
    box = [sx0 + pad, sy0 + pad, sx1 - pad, sy1 - pad]
    for _ in range(3):
        fig.tight_layout(rect=(box[0] / W, box[1] / H, box[2] / W, box[3] / H))
        bb = fig.get_tightbbox(fig.canvas.get_renderer())   # everything drawn so far, in inches
        over = [sx0 + pad - bb.x0, sy0 + pad - bb.y0, bb.x1 - sx1 + pad, bb.y1 - sy1 + pad]
        if max(over) < 0.01:
            break   # labels and legends all fit on the screen
        box = [box[0] + max(over[0], 0), box[1] + max(over[1], 0),
               box[2] - max(over[2], 0), box[3] - max(over[3], 0)]
    fig.add_artist(Rectangle((sx0 / W, sy0 / H), (sx1 - sx0) / W, (sy1 - sy0) / H,
                             transform=fig.transFigure, facecolor=SCREEN,
                             edgecolor='#334155', lw=1, zorder=-10))

    # A faint grid on the screen, about 24 px apart.
    step = 0.24
    segs = [[(a / W, sy0 / H), (a / W, sy1 / H)] for a in np.arange(sx0 + step, sx1, step)]
    segs += [[(sx0 / W, b / H), (sx1 / W, b / H)] for b in np.arange(sy0 + step, sy1, step)]
    fig.add_artist(LineCollection(segs, colors='#E2E8F0', linewidths=0.6,
                                  transform=fig.transFigure, zorder=-9))

    # Top bar with the section label, and a small footer bar.
    for y_bar, height in [(H - top, top), (0, foot)]:
        fig.add_artist(Rectangle((0, y_bar / H), 1, height / H, transform=fig.transFigure,
                                 facecolor=HUD_NAVY, edgecolor='none', zorder=-8))
    fig.add_artist(Line2D([0, 1], [(H - top) / H] * 2, transform=fig.transFigure, color=CYAN, lw=1.5))
    fig.add_artist(Line2D([0, 1], [foot / H] * 2, transform=fig.transFigure, color=CYAN, lw=0.8, alpha=0.6))
    mono = dict(family='monospace', transform=fig.transFigure, va='center')
    fig.text(side / W, 1 - top / (2 * H), 'AMS 261 · M13 · ' + label, color=CYAN,
             fontsize=10, fontweight='bold', ha='left', **mono)
    fig.text(1 - side / W, 1 - top / (2 * H), '● DRONE LINK · LIVE', color=CYAN,
             fontsize=8, ha='right', **mono)
    fig.text(side / W, foot / (2 * H), footer, color=CONTOUR, fontsize=8, ha='left', **mono)
    fig.text(1 - side / W, foot / (2 * H), 'MISSION 13', color=CONTOUR, fontsize=8, ha='right', **mono)

    # Cyan corner brackets just outside the screen.
    off, arm = 0.07, 0.24
    for cx, cy, dx, dy in [(sx0 - off, sy0 - off, 1, 1), (sx1 + off, sy0 - off, -1, 1),
                           (sx0 - off, sy1 + off, 1, -1), (sx1 + off, sy1 + off, -1, -1)]:
        fig.add_artist(Line2D([(cx + dx * arm) / W, cx / W, cx / W],
                              [cy / H, cy / H, (cy + dy * arm) / H],
                              transform=fig.transFigure, color=CYAN, lw=2))
    return fig


CARD = ('max-width:560px;margin-top:6px;border:1px solid #0F1A2B;border-radius:10px;overflow:hidden;'
        'background:#F8FAFC;color:#1E293B;font-family:system-ui,-apple-system,Segoe UI,sans-serif;')
CARD_HEAD = ('background:#0F1A2B;color:#22D3EE;padding:6px 14px;border-bottom:2px solid #22D3EE;'
             'font-family:Menlo,Consolas,monospace;letter-spacing:2px;font-size:12px;')


def check_card(title, rows, summary):
    # A small result card. Each row is (item, mark, words), for example ('range', '✅', 'correct').
    body = ''.join(
        f'<div style="padding:3px 0;color:#1E293B;">'
        f'<span style="display:inline-block;min-width:330px;padding-right:12px;font-family:Menlo,Consolas,monospace;">{item}</span>'
        f'{mark} {words}</div>' for item, mark, words in rows)
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">{title}</div>'
                 f'<div style="padding:8px 14px;color:#1E293B;">{body}'
                 f'<div style="margin-top:6px;font-weight:600;color:#1E293B;">{summary}</div></div></div>'))


def mission_complete_card():
    # Shown once the mission log and the retrieval check are both correct.
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">✅ MISSION 13 COMPLETE</div>'
                 '<div style="padding:8px 14px;line-height:1.5;color:#1E293B;">'
                 'Your mission log and your retrieval check are both correct.<br>'
                 '<b>Prediction for Lecture 14.</b> Can the height approach two different values '
                 'as you walk to the same point along two different paths?<br>'
                 'Write your guess in one sentence. Lecture 14 will test it.</div></div>'))


def hill(X, Y):
    # Height of the hill (hundreds of m) on a numpy grid; nan where there is no reading.
    r = 16 - 4*np.asarray(X, dtype=float)**2 - np.asarray(Y, dtype=float)**2
    return np.where(r >= 0, np.sqrt(np.clip(r, 0, None)), np.nan)


def ellipse_xy(a, b, n=400):
    # Points of the ellipse (x/a)^2 + (y/b)^2 = 1, for plotting.
    s = np.linspace(0, 2*np.pi, n)
    return a*np.cos(s), b*np.sin(s)


def map_axes(ax, title):
    # Equal scales on both axes, so an ellipse is not drawn as a circle.
    ax.set_xlim(-3, 3)
    ax.set_ylim(-4.8, 4.8)
    ax.set_aspect('equal')
    ax.set_xlabel('x (km, east)')
    ax.set_ylabel('y (km, north)')
    ax.axhline(0, color=CONTOUR, lw=0.8, alpha=0.6)
    ax.axvline(0, color=CONTOUR, lw=0.8, alpha=0.6)
    ax.grid(alpha=0.2)
    ax.set_title(title)


print('Setup complete. Widgets available:', HAS_WIDGETS)

---
## 1 · PREDICT · Three drone questions (lecture §4, §8)

🔮 **STAGE 1/7 · PREDICT** ▰▱▱▱▱▱▱

Look at the formula $z=\sqrt{16-4x^2-y^2}$ only. **Do not compute yet.** Make three guesses.

1. At which map points can the drone read a height? (A) At every point. (B) Only inside a circle. (C) Only inside an ellipse.
2. What is the largest height?
3. Is the contour at height $2$ a circle, an ellipse or a line?

Write one sentence of reasoning for each guess. A rough guess is fine. You will check it in Section 4.

In [ ]:
# ✏️ PREDICT. Type your three guesses, then run this cell. Do not compute first.
my_domain_guess = None      # 'A', 'B' or 'C'
my_max_height_guess = None  # a number, in hundreds of meters
my_contour2_guess = None    # 'circle', 'ellipse' or 'line'

if None in (my_domain_guess, my_max_height_guess, my_contour2_guess):
    print('Prediction not logged yet. Type three guesses above and run this cell again.')
else:
    print(f'Prediction logged: domain {my_domain_guess}, largest height {my_max_height_guess}, '
          f'contour at height 2: {my_contour2_guess}.')
    print('You will check it in Section 4.')

In [ ]:
# The hill model. Units: x and y in km, z in hundreds of meters.
f_expr = sp.sqrt(16 - 4*x**2 - y**2)
show_math(r'f(x,y) = \sqrt{16-4x^2-y^2}')
print('sympy stores the same formula as:', f_expr)

---
## 2 · EXPLORE · Safe zone: the domain (lecture §4)

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

The domain is the set of all inputs for which the formula gives a real number.
Our formula has one restriction. A square root needs a nonnegative argument:

$$
16-4x^2-y^2\ge 0 .
$$

The cell below asks sympy to rewrite this restriction. Then it divides by $16$ to read the reach of the region along each axis.

In [ ]:
radicand = 16 - 4*x**2 - y**2
restriction = sp.Ge(radicand, 0)
rewritten = sp.Le(sp.expand(16 - radicand), 16)            # 4x^2 + y^2 <= 16
standard = sp.Le(sp.expand((16 - radicand) / 16), 1)       # x^2/4 + y^2/16 <= 1
show_math(r'16-4x^2-y^2\ge 0\iff ' + sp.latex(rewritten) + r'\iff ' + sp.latex(standard))

# Reach along each axis: put the other variable equal to 0 and solve the boundary equation.
east_reach = max(sp.solve(sp.Eq(radicand.subs(y, 0), 0), x))
north_reach = max(sp.solve(sp.Eq(radicand.subs(x, 0), 0), y))
print(f'Along the x-axis the region reaches x = ±{east_reach} km (east and west).')
print(f'Along the y-axis the region reaches y = ±{north_reach} km (north and south).')

### 2.1 · Probe readings (lecture §3)

The drone tests the map points named on the Lecture 13 page: the summit and the four points where the domain meets the axes.
Before you run the cell, predict the height at each point.

- $S=(0,0)$, $\quad E=(2,0)$, $\quad W=(-2,0)$, $\quad N=(0,4)$, $\quad Q=(0,-4)$.

In [ ]:
probes = {'S': (0, 0), 'E': (2, 0), 'W': (-2, 0), 'N': (0, 4), 'Q': (0, -4)}
print(f"{'point':<14}{'16 - 4x^2 - y^2':>17}   reading")
for name, (px, py) in probes.items():
    r = radicand.subs({x: px, y: py})
    z_val = sp.sqrt(r)
    print(f'{name} = {str((px, py)):<10}{str(r):>17}   z = {z_val}  ({fmt(100*z_val, 0)} m)')
print('Any point outside the ellipse 4x^2 + y^2 = 16 makes 16 - 4x^2 - y^2 negative: no reading there.')

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 7))
X, Y = np.meshgrid(np.linspace(-3, 3, 400), np.linspace(-4.8, 4.8, 400))
ax.contourf(X, Y, 16 - 4*X**2 - Y**2, levels=[0, 17], colors=[GRAD], alpha=0.15)
ex, ey = ellipse_xy(2, 4)
ax.plot(ex, ey, color=ZERO, lw=2.6)
for name, (px, py) in probes.items():
    ax.plot([px], [py], '*' if name == 'S' else 'o', color=INK, ms=13 if name == 'S' else 8)
    ax.annotate(name, (px, py), xytext=(8, 6), textcoords='offset points', color=INK, fontsize=11)
map_axes(ax, 'Section 2 · The domain: where the drone can read a height')
ax.legend(handles=[
    Line2D([0], [0], color=GRAD, alpha=0.3, lw=10, label='blue shading: the domain 4x² + y² ≤ 16'),
    legend_line(ZERO, 'green: the boundary ellipse 4x² + y² = 16 (included)'),
    legend_line(INK, 'black dots: the four reach points, height 0', marker='o'),
    legend_line(INK, 'black star: the summit S, height 4', marker='*'),
], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
hud_frame(fig, 'SEC 2 · SAFE ZONE')
plt.show()

*Figure description.* The map uses equal scales on both axes. The blue region is the domain. It is the inside of the green ellipse $4x^2+y^2=16$ together with the ellipse itself. The region reaches $2$ km east and west and $4$ km north and south. The four black dots $E$, $W$, $N$, $Q$ lie on the edge, where the height is $0$. The black star $S$ is the summit, where the height is $4$. Outside the green ellipse the drone reads nothing.

---
## 3 · EXPLORE · Relief view: the graph and the range (lecture §7)

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

The range is the set of heights that occur. We bound $z$ from both sides.

- A square root is never negative, so $z\ge 0$.
- We have $16-4x^2-y^2\le 16$, so $z\le\sqrt{16}=4$.

Bounds alone do not prove that every height between them occurs. So we also need a point for each height.
The page uses the point $(0,\sqrt{16-z^2})$. The cell below checks this with sympy.

In [ ]:
z0 = sp.symbols('z_0', nonnegative=True)
witness = (0, sp.sqrt(16 - z0**2))                  # the page's point for the height z0
height_there = sp.simplify(f_expr.subs({x: witness[0], y: witness[1]}, simultaneous=True))
show_math(r'f\left(0,\sqrt{16-z_0^2}\right) = ' + sp.latex(height_there)
          + r'\quad\text{for } 0\le z_0\le 4')
print('The witness point gives the height z0:', sp.simplify(height_there - z0) == 0)
print('Largest height, at the summit (0, 0):', f_expr.subs({x: 0, y: 0}))

**The graph.** Square both sides of $z=\sqrt{16-4x^2-y^2}$. We get $4x^2+y^2+z^2=16$ with $z\ge0$.
So the graph is the **upper half** of an ellipsoid. The cell below draws it.

In [ ]:
fig = plt.figure(figsize=(8.5, 6.2))
ax = fig.add_subplot(1, 1, 1, projection='3d')
th, ph = np.meshgrid(np.linspace(0, 2*np.pi, 80), np.linspace(0, np.pi/2, 40))
Xs, Ys, Zs = 2*np.sin(ph)*np.cos(th), 4*np.sin(ph)*np.sin(th), 4*np.cos(ph)   # 4x^2 + y^2 + z^2 = 16, z >= 0
ax.plot_surface(Xs, Ys, Zs, color=GRAD, alpha=0.35, edgecolor=CONTOUR, linewidth=0.3)
ex, ey = ellipse_xy(2, 4)
ax.plot(ex, ey, 0*ex, color=ZERO, lw=2.4)
se = np.linspace(-2, 2, 200)
ax.plot(se, 0*se, np.sqrt(16 - 4*se**2), color=SLICE, lw=2.6)
ax.scatter([0], [0], [4.12], color=INK, s=140, marker='*', depthshade=False)
ax.set_xlim(-2, 2)
ax.set_ylim(-4, 4)
ax.set_zlim(0, 4)
ax.set_xticks([-2, -1, 0, 1, 2])
ax.set_xlabel('x (km, east)')
ax.set_ylabel('y (km, north)')
ax.set_zlabel('z (100 m)')
ax.set_box_aspect((4, 8, 4))
ax.view_init(elev=24, azim=-60)
ax.set_title('Section 3 · The hill: the upper half of 4x² + y² + z² = 16')
ax.legend(handles=[
    Line2D([0], [0], color=GRAD, alpha=0.4, lw=10, label='blue surface: the graph z = f(x, y)'),
    legend_line(ZERO, 'green: the foot of the hill, height 0'),
    legend_line(SLICE, 'purple: the slice y = 0 (east-west profile)'),
    legend_line(INK, 'black star: the summit (0, 0, 4)', marker='*'),
], loc='upper left', bbox_to_anchor=(0.0, 1.0), fontsize=9, frameon=False)
hud_frame(fig, 'SEC 3 · RELIEF VIEW')
plt.show()

*Figure description.* A 3D view of the hill with the same scale on all three axes. Note that one axis unit is $1$ km for $x$ and $y$ but $100$ m for $z$, so the hill looks taller than it is. The blue surface is the graph. Its shadow on the $xy$-plane is the domain of Section 2. The green curve is the foot of the hill, where the height is $0$. The purple curve is the slice $y=0$, the east-west profile. The black star is the summit at height $4$.

---
## 4 · EXPLAIN · Contour map at heights 0, 2 and 4 (lecture §8)

🧠 **STAGE 3/7 · EXPLAIN** ▰▰▰▱▱▱▱

A level curve of $f$ is the set of points in the domain with $f(x,y)=k$. Only heights $k$ in the range $[0,4]$ give a non-empty curve.
For $0\le k\le 4$ we square both sides:

$$
\sqrt{16-4x^2-y^2}=k \iff 4x^2+y^2=16-k^2 .
$$

The cell below solves this with sympy for the three survey heights $k=0,2,4$.
For each ellipse it also checks the answer: it takes points on the ellipse and evaluates $f$ there.

In [ ]:
survey_levels = [0, 2, 4]
level_eq = {}
reach = {}
s_par = sp.symbols('s', real=True)
for kv in survey_levels:
    eq = sp.Eq(4*x**2 + y**2, 16 - kv**2)              # f = k, squared
    # Check 1: squaring is reversible here, because both sides are >= 0 when 0 <= k <= 4.
    same = sp.simplify((f_expr**2 - kv**2) - (16 - kv**2 - (4*x**2 + y**2))) == 0
    a_k, b_k = sp.sqrt(16 - kv**2) / 2, sp.sqrt(16 - kv**2)   # east and north reach
    # Check 2: every point (a_k cos s, b_k sin s) of the ellipse has height k.
    on_curve = sp.simplify(f_expr.subs({x: a_k*sp.cos(s_par), y: b_k*sp.sin(s_par)}, simultaneous=True))
    level_eq[kv], reach[kv] = eq, (a_k, b_k)
    shape = 'the single point (0, 0)' if a_k == 0 else 'an ellipse'
    show_math(r'k=' + str(kv) + r':\quad ' + sp.latex(eq) + r'\quad\text{(' + shape + ')}')
    print(f'   squaring is reversible: {same};  height on the curve: {on_curve};  '
          f'east reach {a_k} ≈ {fmt(a_k)} km, north reach {b_k} ≈ {fmt(b_k)} km')

**Heights outside the range.** Take any height $k>4$. The right side $16-k^2$ is then negative. The left side $4x^2+y^2$ is never negative. So no point satisfies the equation, and the level curve is empty. The next cell checks the sign of the right side.

In [ ]:
k_big = sp.symbols('k', positive=True)
negative_for = sp.solve_univariate_inequality(16 - k_big**2 < 0, k_big, relational=False,
                                            domain=sp.Interval(0, sp.oo))
print('For heights k >= 0, the right side 16 - k^2 is negative exactly when k is in', negative_for)
print('The left side 4x^2 + y^2 is >= 0 for every (x, y). So for k > 4 the level curve is empty.')

### 4.1 · Terrain scanner: choose a height $k$

Move the slider $k$ from $0$ to $4$. The left panel cuts the hill with the plane $z=k$. The right panel shows that cut from above, on the contour map.
The readout gives the east and north reach of the contour.

If the slider does not appear, use the static cell after it. Change `k_static` there and run it again.

In [ ]:
def plot_scanner(k_val=1.0):
    k_val = float(k_val)
    a_k, b_k = math.sqrt(16 - k_val**2) / 2, math.sqrt(16 - k_val**2)
    fig = plt.figure(figsize=(12.5, 5.6))
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax2 = fig.add_subplot(1, 2, 2)

    # Left: the hill and the cutting plane z = k.
    th, ph = np.meshgrid(np.linspace(0, 2*np.pi, 60), np.linspace(0, np.pi/2, 30))
    ax1.plot_surface(2*np.sin(ph)*np.cos(th), 4*np.sin(ph)*np.sin(th), 4*np.cos(ph),
                     color=GRAD, alpha=0.25, edgecolor=CONTOUR, linewidth=0.2)
    PX, PY = np.meshgrid(np.linspace(-2.4, 2.4, 2), np.linspace(-4.4, 4.4, 2))
    ax1.plot_surface(PX, PY, 0*PX + k_val, color=DIRECTION, alpha=0.18)
    cx, cy = ellipse_xy(a_k, b_k)
    ax1.plot(cx, cy, 0*cx + k_val, color=DIRECTION, lw=2.6)
    ax1.set_xlabel('x (km)')
    ax1.set_ylabel('y (km)')
    ax1.set_zlabel('z (100 m)')
    ax1.set_xlim(-2, 2)
    ax1.set_ylim(-4, 4)
    ax1.set_zlim(0, 4)
    ax1.set_xticks([-2, -1, 0, 1, 2])
    ax1.set_box_aspect((4, 8, 4))
    ax1.view_init(elev=22, azim=-60)
    ax1.set_title(f'The hill cut by the plane z = {k_val:.1f}')

    # Right: the contour map with the chosen contour.
    for kk in survey_levels:
        aa, bb = float(reach[kk][0]), float(reach[kk][1])
        col = {0: ZERO, 2: GRAD, 4: INK}[kk]
        if aa > 0:
            ax2.plot(*ellipse_xy(aa, bb), color=col, lw=1.6, ls='--')
    ax2.plot([0], [0], '*', color=INK, ms=13)
    ax2.plot(cx, cy, color=DIRECTION, lw=2.8)
    map_axes(ax2, f'Contour at height k = {k_val:.1f} (amber)')
    ax2.legend(handles=[
        legend_line(DIRECTION, 'amber: the contour at your height k'),
        legend_line(ZERO, 'green dashed: height 0', ls='--'),
        legend_line(GRAD, 'blue dashed: height 2', ls='--'),
        legend_line(INK, 'black star: height 4 (the summit)', marker='*'),
        Line2D([0], [0], color=GRAD, alpha=0.3, lw=8, label='left, blue: the hill surface'),
    ], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
    hud_frame(fig, 'SEC 4.1 · TERRAIN SCANNER')
    display(fig)      # display, then close: works the same with or without the slider
    plt.close(fig)


if HAS_WIDGETS:
    k_slider = SelectionSlider(options=[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.5, 4.0], value=1.0,
                               description='k (100 m)', continuous_update=False,
                               layout=Layout(width='100%'),
                               style={'handle_color': CYAN, 'description_width': 'initial'})
    readout = WidgetHTML()

    def update_readout(change=None):
        kv = k_slider.value
        a_k, b_k = math.sqrt(16 - kv**2) / 2, math.sqrt(16 - kv**2)
        readout.value = (f'<span style="font-family:Menlo,Consolas,monospace;">READOUT · '
                         f'height k = {kv:.1f} ({100*kv:.0f} m) · east reach {a_k:.3f} km · '
                         f'north reach {b_k:.3f} km</span>')

    k_slider.observe(update_readout, names='value')
    update_readout()
    scanner_screen = interactive_output(plot_scanner, {'k_val': k_slider})
    display(VBox([k_slider, scanner_screen, readout]))
else:
    print('ipywidgets is not available here. Use the static cell below instead.')

*Figure description.* Left: the hill (blue) and the horizontal plane $z=k$ (amber). They meet in an amber ellipse. Right: the same ellipse seen from above, on a map with equal scales. The dashed green ellipse is the contour at height $0$. The dashed blue ellipse is the contour at height $2$. The black star is the summit, the contour at height $4$. As $k$ grows, the amber ellipse shrinks toward the star.

In [ ]:
# Static version (no widgets needed). Change k_static (between 0 and 4) and run again.
k_static = 1
plot_scanner(k_static)

*Figure description.* The same two panels as above, drawn once at the height stored in `k_static`.

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🧠 Explain.</strong> Every contour of the hill has the form $4x^2+y^2=c$. Explain <b>in one sentence</b> why the contours are ellipses and not circles.
<br><br>Then answer: can two contours of different heights cross? Answer yes or no, and give one sentence of reason. Use the definition of a function.

</div>

Write your sentences first. Then run the next cell to compare the answers with your Section 1 prediction.

In [ ]:
# Check your Section 1 prediction.
print('Domain: C. The drone reads a height only inside and on the ellipse 4x^2 + y^2 <= 16.')
print('Largest height: 4 (400 m), at the summit (0, 0).')
print('Contour at height 2: an ellipse, 4x^2 + y^2 = 12.')
print()
if None in (my_domain_guess, my_max_height_guess, my_contour2_guess):
    print('You did not log a prediction in Section 1. Next time, commit a guess first.')
else:
    print('Your domain guess:', my_domain_guess, '| correct:', str(my_domain_guess).strip().upper() == 'C')
    print('Your largest height:', my_max_height_guess, '| correct:', check_answer(my_max_height_guess, 4))
    print('Your contour shape:', my_contour2_guess, '| correct:',
          str(my_contour2_guess).strip().lower() == 'ellipse')

---
## 5 · CHALLENGE · Contour spacing and steepness (lecture §8)

🎯 **STAGE 4/7 · CHALLENGE** ▰▰▰▰▱▱▱

The heights $0$, $2$, $4$ are equally spaced. So close contours mean a steep slope, and wide gaps mean a gentle one.
The cell below measures the gaps between neighboring contours along two lines from the summit: toward the east ($y=0$) and toward the north ($x=0$).

In [ ]:
gap_east_02 = sp.simplify(reach[0][0] - reach[2][0])
gap_north_02 = sp.simplify(reach[0][1] - reach[2][1])
gap_east_24 = sp.simplify(reach[2][0] - reach[4][0])
show_math(r'\text{east gap, heights 0 and 2: } ' + sp.latex(gap_east_02) + r'\approx ' + fmt(gap_east_02, 3)
          + r'\text{ km},\qquad \text{north gap, heights 0 and 2: } ' + sp.latex(gap_north_02)
          + r'\approx ' + fmt(gap_north_02, 3) + r'\text{ km}')
show_math(r'\text{east gap, heights 2 and 4: } ' + sp.latex(gap_east_24) + r'\approx ' + fmt(gap_east_24, 3)
          + r'\text{ km}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={'width_ratios': [1.35, 1]})

# Left: the two profiles through the summit.
se = np.linspace(-2, 2, 801)
sn = np.linspace(-4, 4, 801)
ax1.plot(se, np.sqrt(np.clip(16 - 4*se**2, 0, None)), color=SLICE, lw=2.6)
ax1.plot(sn, np.sqrt(np.clip(16 - sn**2, 0, None)), color=CONTOUR, lw=2.2)
for kk, col in [(0, ZERO), (2, GRAD), (4, INK)]:
    ax1.axhline(kk, color=col, lw=1, ls=':')
for kk in [0, 2]:
    ax1.plot([float(reach[kk][0])], [kk], 'o', color=SLICE, ms=7)
    ax1.plot([float(reach[kk][1])], [kk], 's', color=CONTOUR, ms=7)
ax1.set_xlim(-4.3, 4.3)
ax1.set_ylim(-0.2, 5.9)
ax1.set_yticks(range(5))
ax1.set_xlabel('distance from the summit along the line (km)')
ax1.set_ylabel('height z (100 m)')
ax1.set_title('Profiles of the hill through the summit')
ax1.grid(alpha=0.3)
ax1.legend(handles=[
    legend_line(SLICE, 'purple: east-west profile (y = 0)'),
    legend_line(CONTOUR, 'grey: north-south profile (x = 0)'),
    legend_line(INK, 'dotted lines: heights 0 (green), 2 (blue), 4 (black)', ls=':'),
    legend_line(SLICE, 'purple dots: east reach at heights 2 and 0', marker='o'),
    legend_line(CONTOUR, 'grey squares: north reach at heights 2 and 0', marker='s'),
], loc='upper center', ncol=2, fontsize=8)

# Right: the contour map, east half, with the two gaps marked.
for kk, col in [(0, ZERO), (2, GRAD)]:
    ax2.plot(*ellipse_xy(float(reach[kk][0]), float(reach[kk][1])), color=col, lw=2.2)
ax2.plot([0], [0], '*', color=INK, ms=13)
ax2.plot([float(reach[2][0]), 2], [0, 0], color=SLICE, lw=5, solid_capstyle='butt')
ax2.plot([0, 0], [float(reach[2][1]), 4], color=DIRECTION, lw=5, solid_capstyle='butt')
map_axes(ax2, 'Gaps between the contours at heights 0 and 2')
ax2.legend(handles=[
    legend_line(ZERO, 'green: contour at height 0'),
    legend_line(GRAD, 'blue: contour at height 2'),
    legend_line(SLICE, 'thick purple: east gap'),
    legend_line(DIRECTION, 'thick amber: north gap'),
    legend_line(INK, 'black star: summit, height 4', marker='*'),
], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
hud_frame(fig, 'SEC 5 · CONTOUR SPACING')
plt.show()

*Figure description.* Left: two profiles of the hill through the summit. The purple curve is the east-west profile. The grey curve is the north-south profile. Dotted lines mark the heights $0$, $2$ and $4$. The purple curve falls from height $2$ to height $0$ over a much shorter distance. Right: the contour map with equal scales. The short thick purple bar is the east gap between the contours at heights $0$ and $2$. The longer thick amber bar is the north gap.

<div style="background:#FFFBEB;border-left:6px solid #D97706;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🎯 Challenge question.</strong> Answer in two or three sentences.

1. Compare the east gap and the north gap between the contours at heights $0$ and $2$. On which side is the hill steeper?
2. Now stay on the east side. Compare the gap between heights $0$ and $2$ with the gap between heights $2$ and $4$. Is the hill steepest near its foot or near its top?
3. The drone flies straight north from the summit to the edge of the hill. Does its height drop faster at the start of the flight or near the end? Use the grey profile.

</div>

<details>
<summary>Hint (open after you try)</summary>

Compare the lengths of the thick purple bar and the thick amber bar in the right panel.
For part 3: where is the grey curve steepest?

</details>

---
## 6 · TRANSFER · Shell scan: level surfaces (lecture §11)

🚀 **STAGE 5/7 · TRANSFER** ▰▰▰▰▰▱▱

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🚀 Transfer, from curves to surfaces.</strong> Page Example 13 uses the function of three variables
$$
F(x,y,z)=4x^2+y^2+z^2 .
$$
Its level surfaces are $F(x,y,z)=c$: no point for $c<0$, one point for $c=0$, and ellipsoids for $c>0$.

1. In Section 3 we squared the hill's formula. Which level surface $F=c$ contains the graph of the hill? What is $c$?
2. Write the semi-axes of the ellipsoid $F=4$. (This is the shell of the page's Shell Scanner.)

</div>

In [ ]:
# ✏️ Optional check for part 1. Type your value of c, then run.
my_c = None

F_expr = 4*x**2 + y**2 + z**2
on_graph = sp.simplify(F_expr.subs(z, f_expr))      # F evaluated on the graph z = f(x, y)
if my_c is None:
    print('Type your value of c first.')
else:
    print('F on the graph of the hill is constant and equal to your c:', check_answer(my_c, on_graph))

<details>
<summary>Check your answer</summary>

1. On the graph we have $z^2=16-4x^2-y^2$. So $F=4x^2+y^2+z^2=16$ at every point of the graph. The hill is the upper half ($z\ge0$) of the level surface $F=16$.
2. Divide $4x^2+y^2+z^2=4$ by $4$: $\ x^2+\tfrac{y^2}{4}+\tfrac{z^2}{4}=1$. The semi-axes are $1$, $2$ and $2$.

</details>

---
## 7 · CREATE · Mission log

📝 **STAGE 6/7 · CREATE** ▰▰▰▰▰▰▱

Fill in the log with your own results from Sections 2–5. Use km for distances and hundreds of meters for heights.
Then write one sentence in your own words.

In [ ]:
# ✏️ CREATE. Fill in your mission log, then run this cell and the next one.
mission_log = {
    'east_reach': None,     # how far east of the summit the domain reaches (km)
    'north_reach': None,    # how far north of the summit the domain reaches (km)
    'range': [None, None],  # the smallest and the largest height, [low, high]
    'c_height_2': None,     # the contour at height 2 is 4x^2 + y^2 = c. What is c?
    'gap_ratio': None,      # north gap / east gap between the contours at heights 0 and 2
    'one_sentence': '',     # why is the hill steeper toward the east than toward the north?
}

In [ ]:
# Check the mission log. The card shows only "correct" or "try again", never the expected values.
targets = {
    'east_reach': east_reach,
    'north_reach': north_reach,
    'range': [0, f_expr.subs({x: 0, y: 0})],
    'c_height_2': level_eq[2].rhs,
    'gap_ratio': sp.simplify(gap_north_02 / gap_east_02),
}
checks = [(key, check_answer(mission_log[key], target)) for key, target in targets.items()]
rows = [(key, '✅', 'correct') if ok else (key, '🔁', 'try again') for key, ok in checks]
sentence = mission_log['one_sentence'].strip()
rows.append(('one_sentence', '✅', 'logged') if sentence else ('one_sentence', '✏️', 'write one sentence'))

n_ok = sum(ok for _, ok in checks)
mission_log_ok = n_ok == len(checks) and bool(sentence)
summary = f'{n_ok} of {len(checks)} correct.'
summary += ' Sentence logged.' if sentence else ' Sentence not written yet.'
if mission_log_ok:
    summary += ' Mission log complete. Next: the debrief.'
check_card('MISSION LOG · CHECK', rows, summary)
if mission_log_ok and globals().get('quiz_ok', False):
    mission_complete_card()

---
## 8 · DEBRIEF · ⚠️ Common errors: check yourself against these

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#FEF2F2;border-left:6px solid #DC2626;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>⚠️ Common errors.</strong>

- **Domain and range mixed up.** The domain is a set of points in $\mathbb R^2$: here the region $4x^2+y^2\le16$. The range is a set of numbers in $\mathbb R$: here the interval $[0,4]$.
- **Drawing the domain as a circle.** The coefficients of $x^2$ and $y^2$ differ. So the region is an ellipse, $2$ km wide to each side and $4$ km to the north and south. Use equal scales on both axes, or the picture lies.
- **Bounds without a witness.** $0\le z\le4$ is only half of the range argument. You must also show that every height in $[0,4]$ occurs (Section 3).
- **Not checking $k$ against the range.** For a height $k>4$ the equation $4x^2+y^2=16-k^2$ has a negative right side. The level curve is empty (Section 4).
- **Losing a sign restriction when squaring.** $z=\sqrt{16-4x^2-y^2}$ gives $4x^2+y^2+z^2=16$ **and** $z\ge0$. The hill is a half ellipsoid, not the whole one.
- **Reading steepness from unequal height steps.** Contour spacing shows steepness only when the heights are equally spaced, like $0$, $2$, $4$.

</div>

## 9 · DEBRIEF · Retrieval check: mini-quiz

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

These questions use only examples from the Lecture 13 page. Fill in `student_answers` from memory, then run the cell. Numbers may be decimals (three digits are enough) or exact strings such as `"2*sqrt(2)"`.
The check card shows only **correct** or **try again**. After your second try you may set `reveal = True` to see the key.

In [ ]:
# ✏️ Mini-quiz on the Lecture 13 page.
student_answers = {
    'ex6_range': None,               # Example 6: range of sqrt(25 - x^2 - 4y^2), as [low, high]
    'ex9_radius_at_k_2': None,       # Example 9: radius of the level circle of sqrt(9 - x^2 - y^2) at k = 2
    'ex10_x_intercept_k_5': None,    # Example 10: positive x-intercept of the level curve of 4x^2 + y^2 + 1 at k = 5
    'ex14_f_at_1_1_1': None,         # Example 14: f(1, 1, 1) for sqrt(x) + sqrt(y) + sqrt(z) + ln(4 - x^2 - y^2 - z^2)
    'level_curves_can_cross': None,  # can two level curves with different k cross? True or False
}
reveal = False   # set to True only after your second try

_key = json.loads(base64.b64decode('eyJleDZfcmFuZ2UiOiBbMCwgNV0sICJleDlfcmFkaXVzX2F0X2tfMiI6IDIuMjM2MDY3OTc3NDk5NzksICJleDEwX3hfaW50ZXJjZXB0X2tfNSI6IDEsICJleDE0X2ZfYXRfMV8xXzEiOiAzLCAibGV2ZWxfY3VydmVzX2Nhbl9jcm9zcyI6IGZhbHNlfQ=='))
if any(v is not None for v in student_answers.values()):
    quiz_attempts = globals().get('quiz_attempts', 0) + 1
else:
    quiz_attempts = globals().get('quiz_attempts', 0)

n_correct = 0
rows = []
for name, target in _key.items():
    ok = check_answer(student_answers[name], target)
    n_correct += ok
    rows.append((name, '✅', 'correct') if ok else (name, '🔁', 'try again'))
quiz_ok = n_correct == len(_key)
check_card('RETRIEVAL · CHECK', rows, f'{n_correct} of {len(_key)} correct · attempt {quiz_attempts}')

if reveal and quiz_attempts < 2:
    print('Try at least twice before you reveal the key.')
elif reveal:
    print('Key:', _key)
elif quiz_ok and globals().get('mission_log_ok', False):
    mission_complete_card()
elif quiz_ok:
    print('All correct. Finish the mission log in Stage 6 to complete the mission.')

## 10 · DEBRIEF · 🤖 AI Learning Companion

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:6px 14px;border-radius:8px;color:#1E293B;">
<strong>🤖 Student-facing AI contract — paste this at the start of every AI session</strong>
<br>1. Ask me what I have already tried before you explain anything.
<br>2. Do not give a complete solution until I have made a prediction or proposed a next step.
<br>3. Point to the first gap in my reasoning, not every later consequence.
<br>4. Make me verify the result by units, sign, substitution, a graph, or a limiting case.
<br>5. End with one transfer question that changes the context.
</div>

<div style="background:#F8FAFC;border-left:6px solid #94A3B8;padding:6px 14px;border-radius:8px;margin-top:6px;color:#1E293B;">
<strong>🔎 Verification rule.</strong> Treat an AI answer as a conjecture. Check it against the definitions, theorems and figures on this page. Bring any disagreement to class. An AI answer never overrides a checked definition or proof. Neither does an illustration.
</div>

**Prompt for this lab: Error Hunter.** Paste your own domain, range or level-curve solution in place of `[paste my work]`.

```
Here is my solution to a domain, range, graph, or level-curve problem:
[paste my work]
Find the FIRST invalid step. Do not fix it. Instead:
(1) quote that step, (2) name the underlying misconception in one sentence,
(3) give me a two-line counterexample or sanity check that exposes it,
(4) ask me to repair the step myself.
If my solution is correct, instead give me a harder variant that breaks the
method I used, and ask me why it breaks.
```

## 11 · DEBRIEF · 🔒 Independence check

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:8px 16px;border-radius:8px;color:#1E293B;">
<strong>🔒 Independence rule.</strong> <i>No AI, no hints, new numbers. This shows that you can do it alone.</i>
</div>

Let $g(x,y)=\ln(25-x^2-y^2)$.

1. Find the domain of $g$ and sketch it. Is the boundary included?
2. Find the range of $g$.
3. Find and sketch the level curve $g(x,y)=\ln 16$.

Do this on paper, without AI and without the worked examples. It is not checked here. It is practice for Exam 2.

---
## 12 · OPTIONAL · Review examples from the lecture page

**OPTIONAL · outside the seven stages.** Skip this part if you are short of time. The mission ends with the prediction for Lecture 14 below.

Try each example on paper before you run its cell.

### 12.1 · Example 12 · Doubling labor and capital (lecture §10)

A toy manufacturer estimates $f(x,y)=100x^{0.6}y^{0.4}$. Here $x$ is labor and $y$ is capital.
Compare the outputs at $(1000,500)$ and $(2000,1000)$.

In [ ]:
cobb = lambda L, K: 100 * sp.Integer(L)**sp.Rational(3, 5) * sp.Integer(K)**sp.Rational(2, 5)
out1, out2 = cobb(1000, 500), cobb(2000, 1000)
print(f'f(1000, 500)  ≈ {float(out1):,.0f} units')
print(f'f(2000, 1000) ≈ {float(out2):,.0f} units')
print('Ratio f(2000, 1000) / f(1000, 500) =', sp.simplify(out2 / out1))
lam = sp.symbols('lambda', positive=True)
Lp, Kp = sp.symbols('L K', positive=True)
scaled = sp.simplify(100*(lam*Lp)**sp.Rational(3, 5)*(lam*Kp)**sp.Rational(2, 5)
                     / (100*Lp**sp.Rational(3, 5)*Kp**sp.Rational(2, 5)))
show_math(r'\frac{f(\lambda x,\lambda y)}{f(x,y)} = ' + sp.latex(scaled))

*Note.* The exponents add to $0.6+0.4=1$. So scaling both inputs by $\lambda$ scales the output by $\lambda$. Economists call this constant returns to scale. The page rounds the two outputs to $75{,}800$ and $151{,}600$ units.

### 12.2 · Example 4(b) · The domain of $\ln(xy)$ (lecture §4)

We need $xy>0$. Which quadrants qualify? Is the point $(-1,-1)$ in the domain?

In [ ]:
g4 = sp.log(x*y)
for pt in [(1, 1), (-1, -1), (1, -1), (-1, 1)]:
    prod = pt[0] * pt[1]
    status = f'in the domain, value ln({prod}) = {sp.log(prod)}' if prod > 0 else 'not in the domain'
    print(f'(x, y) = {pt}:  xy = {prod:>2}  ->  {status}')
print('So the domain is the open first and third quadrants: x and y have the same sign.')

---
## 13 · DEBRIEF · Prediction for Lecture 14

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

The mission is complete when the Stage 6 log and the retrieval check both show ✅ for every item.
Your last task is one prediction.

<div style="background:#0A0E14;border:1px solid #22D3EE;border-radius:10px;padding:10px 16px;color:#E2E8F0;">
<span style="color:#22D3EE;font-family:Menlo,Consolas,monospace;letter-spacing:1px;font-size:12px;">MISSION 13 · FINAL PROMPT · PREDICTION FOR LECTURE 14</span><br>
Lecture 14 is about limits and continuity. In one variable you can approach a point from only two sides. On a map you can walk to a point along infinitely many paths.
<br>Can the height approach two different values as you walk to the same point along two different paths?
<br>Write your guess in one sentence. Lecture 14 will test it.
</div>

---
<div style="background:#0F1A2B;border-top:2px solid #22D3EE;color:#94A3B8;padding:8px 16px;font-family:Menlo,Consolas,monospace;font-size:12px;letter-spacing:1px;">
END OF LAB 13 · HILL SURVEY · © Tan H. Cao, for AMS 261 students
</div>